# 🚗 Honda HR-V EXL 2017 — Buscador de Oportunidades
**12 sites** | Cor: sem branco | Local: SP + até 1h15 | Scoring completo de custo-benefício | Detector de golpes


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install",
    "requests", "pandas", "beautifulsoup4", "lxml", "-q"])
print("✓ Dependências prontas")


In [ ]:
# ── Cole aqui TODO o conteúdo de honda_hrv_search.py (exceto o bloco if __name__) ──
# OU simplesmente importe:
import importlib.util, sys, os
spec = importlib.util.spec_from_file_location(
    "hrv", os.path.join(os.path.dirname(os.path.abspath("__file__")), "honda_hrv_search.py")
)
hrv = importlib.util.module_from_spec(spec)
spec.loader.exec_module(hrv)
print("✓ Script carregado")


In [ ]:
# ── Busca preço FIPE ──────────────────────────────────────────────────────────
fipe_ref = hrv.buscar_preco_fipe()


In [ ]:
# ── Busca em todos os sites ───────────────────────────────────────────────────
print("Buscando em 12 sites...")
todos = []
todos += hrv.buscar_mercadolivre()
todos += hrv.buscar_webmotors()
todos += hrv.buscar_olx()
todos += hrv.buscar_icarros()
todos += hrv.buscar_kavak()
todos += hrv.buscar_mobiauto()
todos += hrv.buscar_vrum()
todos += hrv.buscar_autoline()
todos += hrv.buscar_localiza_seminovos()
todos += hrv.buscar_movida_seminovos()
todos += hrv.buscar_honda_seminovos()
todos += hrv.buscar_autoshow()
print(f"\nTotal coletado (com duplicatas): {len(todos)}")


In [ ]:
# ── Deduplica ─────────────────────────────────────────────────────────────────
vistos, unicos = set(), []
for item in todos:
    chave = item.get("link","").split("?")[0].rstrip("/")
    if chave and chave not in vistos:
        vistos.add(chave); unicos.append(item)
print(f"Únicos: {len(unicos)}")


In [ ]:
# ── Aplica filtros ────────────────────────────────────────────────────────────
motivos = {"versao":0,"ano":0,"cor":0,"cidade":0,"suspeito":0}
validos, alertas = [], []

for item in unicos:
    desc = str(item.get("titulo","")) + " " + str(item.get("descricao",""))
    if not hrv.eh_exl(item.get("titulo",""), item.get("versao")):
        motivos["versao"] += 1; continue
    if not hrv.eh_ano_certo(item.get("titulo",""), item.get("ano")):
        motivos["ano"] += 1; continue
    if not hrv.cor_ok(item.get("cor")):
        motivos["cor"] += 1; continue
    if not hrv.cidade_ok(item.get("cidade"), item.get("estado")):
        motivos["cidade"] += 1; continue
    _, _, scam = hrv.detectar_flags(desc)
    alerta_preco = hrv.preco_suspeito(item.get("preco"), fipe_ref)
    if scam or alerta_preco == "scam":
        motivos["suspeito"] += 1; continue
    item["_score"] = hrv.score_listing(item, fipe_ref)
    item["_alerta"] = alerta_preco == "alerta"
    (alertas if alerta_preco == "alerta" else validos).append(item)

ranking = sorted(validos, key=lambda x: x["_score"], reverse=True)
ranking_alertas = sorted(alertas, key=lambda x: x["_score"], reverse=True)

print("Filtros aplicados:")
for k,v in motivos.items(): print(f"  {k:12}: -{v}")
print(f"\nAnúncios válidos : {len(ranking)}")
print(f"Com alerta preço : {len(ranking_alertas)}")


In [ ]:
# ── Exibe os resultados ───────────────────────────────────────────────────────
print(f"FIPE referência: {hrv.fmt_price(fipe_ref)}\n")
for i, item in enumerate(ranking[:20], 1):
    hrv.imprimir_item(i, item, fipe_ref)


In [ ]:
# ── Anúncios com alerta de preço (verificar antes de contatar) ───────────────
if ranking_alertas:
    print("⚠ PREÇO MUITO ABAIXO DA FIPE — verifique antes de contatar\n")
    for i, item in enumerate(ranking_alertas[:5], 1):
        hrv.imprimir_item(i, item, fipe_ref)


In [ ]:
# ── Tabela pandas ────────────────────────────────────────────────────────────
import pandas as pd
todos_validos = ranking + ranking_alertas
if todos_validos:
    df = pd.DataFrame(todos_validos)
    df["preco_fmt"] = df["preco"].apply(hrv.fmt_price)
    df["km_fmt"]    = df["km"].apply(hrv.fmt_km)
    df["alerta"]    = df.get("_alerta", False)
    cols = ["fonte","titulo","preco_fmt","km_fmt","ano","versao","cor","cidade","_score","alerta","link"]
    cols = [c for c in cols if c in df.columns]
    display(
        df[cols].rename(columns={"_score":"score"})
          .sort_values("score", ascending=False)
          .reset_index(drop=True)
          .style.background_gradient(subset=["score"], cmap="RdYlGn")
          .set_caption(f"Honda HR-V EXL {hrv.ANO_ALVO} — melhores oportunidades")
    )


In [ ]:
# ── Estatísticas ──────────────────────────────────────────────────────────────
if todos_validos:
    precos = [x["preco"] for x in todos_validos if x.get("preco")]
    kms    = [hrv.parse_km(x["km"]) for x in todos_validos if hrv.parse_km(x.get("km"))]
    print("RESUMO")
    print(f"  Preço médio : {hrv.fmt_price(int(sum(precos)/len(precos))) if precos else 'N/I'}")
    print(f"  Mais barato : {hrv.fmt_price(min(precos)) if precos else 'N/I'}")
    print(f"  Ref. FIPE   : {hrv.fmt_price(fipe_ref)}")
    print(f"  KM médio    : {hrv.fmt_km(int(sum(kms)/len(kms))) if kms else 'N/I'}")
    print(f"  Menor KM    : {hrv.fmt_km(min(kms)) if kms else 'N/I'}")
